# IN4640 Assignment 2 — Question 1: Line Fitting & Alignment

**Name:** *Gayashan W.J*  
**Index:** *215525P*  

---

## Question
> Table 1 shows the first five lines of the dataset in the file lines.csv. It represents three
points scatters conforming to three lines. The coordinates of the points are stored in columns
x1, x2, x3, y1, y2, and y3.
>
> **(a)** Use **Total Least Squares (TLS)** to fit a line using only the data corresponding to the **first line**. Report the resulting parameters.  
> **(b)** Now, use all the points as indicate in the code snippet below and fit three lines. Hint:
Run RANSACtofindaline, mask the consensus and run again and so on to find the
three lines

---

## Theory

### Total Least Squares (TLS)
Unlike Ordinary Least Squares (OLS) which minimises vertical residuals, TLS minimises **perpendicular (orthogonal) distances** from points to the line. This is appropriate when both $x$ and $y$ have measurement noise.

Given points $(x_i, y_i)$, we:
1. Centre the data: $\tilde{x}_i = x_i - \bar{x}$, $\tilde{y}_i = y_i - \bar{y}$
2. Form the data matrix $A = [\tilde{x} \; \tilde{y}]$
3. Compute SVD: $A = U \Sigma V^T$
4. The **normal vector** $(a, b)$ of the line is the **last row of $V^T$** (right singular vector for smallest singular value)
5. Line equation: $ax + by = c$ where $c = a\bar{x} + b\bar{y}$

### RANSAC (Random Sample Consensus)
RANSAC robustly fits a model in the presence of outliers:
1. Randomly sample the minimum number of points (2 for a line)
2. Fit a candidate model
3. Count **inliers** (points within threshold distance)
4. Repeat $N$ iterations, keep the best model
5. Refit using all inliers of the best model

For three lines: run RANSAC → remove inliers → run again → repeat.


In [9]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # remove this line if running interactively

# ── Load data ──────────────────────────────────────────────────────────────────
D = np.genfromtxt("lines.csv", delimiter=",", skip_header=1)
X_cols = D[:, :3]
Y_cols = D[:, 3:]
X_all = X_cols.flatten()
Y_all = Y_cols.flatten()

print(f"Dataset shape: {D.shape}  →  {len(X_all)} total points after flattening")
print(f"\nFirst 3 rows:")
print(f"{'x1':>10} {'x2':>10} {'x3':>10} {'y1':>10} {'y2':>10} {'y3':>10}")
for row in D[:3]:
    print("  ".join(f"{v:>10.4f}" for v in row))


Dataset shape: (100, 6)  →  300 total points after flattening

First 3 rows:
        x1         x2         x3         y1         y2         y3
   -5.3055     -4.0601     -5.2613    -12.6663     -3.7962      3.6917
   -5.5404     -5.0032     -3.9926    -11.0077     -3.9856      4.9000
   -4.9821     -4.5845     -4.3312    -11.6973     -3.5893      5.0469


---
## Part (a): TLS on Line 1 Only


In [10]:
# ── Extract Line 1 data (columns x1, y1) ──────────────────────────────────────
x1 = D[:, 0]
y1 = D[:, 3]

print(f"Line 1: {len(x1)} points")
print(f"x range: [{x1.min():.3f}, {x1.max():.3f}]")
print(f"y range: [{y1.min():.3f}, {y1.max():.3f}]")


Line 1: 100 points
x range: [-5.540, 5.740]
y range: [-12.666, 0.203]


In [11]:
# ── Total Least Squares via SVD ────────────────────────────────────────────────
# Step 1: Centre the data
xm, ym = x1.mean(), y1.mean()
A = np.column_stack([x1 - xm, y1 - ym])

# Step 2: SVD
U, S, Vt = np.linalg.svd(A)
print(f"Singular values: {S}")

# Step 3: Normal vector = last row of Vt (smallest singular value)
a, b = Vt[-1]
c = a * xm + b * ym

# Ensure unit normal
norm = np.sqrt(a**2 + b**2)
a, b, c = a/norm, b/norm, c/norm

# Convert to slope-intercept: y = mx + q
m_tls = -a / b
q_tls =  c / b

print("\n" + "="*50)
print("TLS Result — Line 1")
print("="*50)
print(f"  Normal form:     {a:.6f}·x + {b:.6f}·y = {c:.6f}")
print(f"  Slope-intercept: y = {m_tls:.6f}·x + ({q_tls:.6f})")
print(f"  Normal vector:   (a={a:.6f}, b={b:.6f})")
print(f"  Offset:          c = {c:.6f}")


Singular values: [45.53806035  5.0137849 ]

TLS Result — Line 1
  Normal form:     0.773562·x + -0.633721·y = 3.794192
  Slope-intercept: y = 1.220666·x + (-5.987165)
  Normal vector:   (a=0.773562, b=-0.633721)
  Offset:          c = 3.794192


In [12]:
# ── Visualise TLS fit ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(x1, y1, color='steelblue', s=25, alpha=0.7, label='Line 1 data points')

xr = np.linspace(x1.min()-1, x1.max()+1, 300)
yr = m_tls * xr + q_tls
ax.plot(xr, yr, 'r-', lw=2.5, label=f'TLS fit: y = {m_tls:.3f}x + ({q_tls:.3f})')

# Draw a few perpendicular distance lines to illustrate TLS
for xi, yi in zip(x1[::8], y1[::8]):
    # Foot of perpendicular from (xi,yi) to line ax+by=c
    foot_x = xi - a*(a*xi + b*yi - c)
    foot_y = yi - b*(a*xi + b*yi - c)
    ax.plot([xi, foot_x], [yi, foot_y], 'g-', alpha=0.3, lw=0.8)

ax.set_title('Part (a): Total Least Squares — Line 1\n(green lines show perpendicular distances)', fontsize=13, fontweight='bold')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('q1a_tls.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved q1a_tls.png")


Saved q1a_tls.png


In [13]:
# ── Helper functions ──────────────────────────────────────────────────────────
def fit_line_tls(x, y):
    """Fit a line to points using TLS (SVD). Returns (a, b, c) with a²+b²=1."""
    xm, ym = x.mean(), y.mean()
    A = np.column_stack([x - xm, y - ym])
    _, _, Vt = np.linalg.svd(A)
    a, b = Vt[-1]
    c = a * xm + b * ym
    norm = np.sqrt(a**2 + b**2)
    return a/norm, b/norm, c/norm

def perpendicular_distance(x, y, a, b, c):
    """Perpendicular distance from each point to line ax+by=c."""
    return np.abs(a*x + b*y - c)

def ransac_line(x, y, n_iter=1000, threshold=0.5, seed=42):
    """
    RANSAC line fitting.
    Returns: (a, b, c, inlier_mask)
    """
    best_inliers = None
    best_score = 0
    rng = np.random.default_rng(seed)

    for _ in range(n_iter):
        # Sample 2 random points
        idx = rng.choice(len(x), 2, replace=False)
        try:
            a, b, c = fit_line_tls(x[idx], y[idx])
        except Exception:
            continue

        # Count inliers
        dists = perpendicular_distance(x, y, a, b, c)
        inliers = dists < threshold

        if inliers.sum() > best_score:
            best_score = inliers.sum()
            best_inliers = inliers

    # Refit on full inlier set for best accuracy
    a, b, c = fit_line_tls(x[best_inliers], y[best_inliers])
    return a, b, c, best_inliers

print("Helper functions defined: fit_line_tls(), perpendicular_distance(), ransac_line()")


Helper functions defined: fit_line_tls(), perpendicular_distance(), ransac_line()


In [14]:
# ── Run RANSAC iteratively for 3 lines ────────────────────────────────────────
remaining_x   = X_all.copy()
remaining_y   = Y_all.copy()
remaining_idx = np.arange(len(X_all))

lines      = []
inlier_idx = []

print("="*55)
print("RANSAC: Finding 3 Lines")
print("="*55)

for i in range(3):
    a, b, c, inliers_local = ransac_line(remaining_x, remaining_y, seed=42+i)
    m_line = -a / b
    q_line =  c / b

    print(f"\nLine {i+1}:")
    print(f"  Normal form:     {a:.6f}·x + {b:.6f}·y = {c:.6f}")
    print(f"  Slope-intercept: y = {m_line:.6f}·x + ({q_line:.6f})")
    print(f"  Inliers found:   {inliers_local.sum()} / {len(remaining_x)} remaining points")

    lines.append((a, b, c))
    inlier_idx.append(remaining_idx[inliers_local])

    # Remove inliers before next iteration
    mask = ~inliers_local
    remaining_x   = remaining_x[mask]
    remaining_y   = remaining_y[mask]
    remaining_idx = remaining_idx[mask]

print(f"\nPoints unassigned after 3 rounds: {len(remaining_x)}")


RANSAC: Finding 3 Lines

Line 1:
  Normal form:     -0.413720·x + -0.910404·y = -1.900344
  Slope-intercept: y = -0.454436·x + (2.087364)
  Inliers found:   84 / 300 remaining points

Line 2:
  Normal form:     0.735298·x + -0.677744·y = -0.894100
  Slope-intercept: y = 1.084920·x + (1.319230)
  Inliers found:   65 / 216 remaining points

Line 3:
  Normal form:     -0.789338·x + 0.613959·y = -3.713373
  Slope-intercept: y = 1.285653·x + (-6.048243)
  Inliers found:   64 / 151 remaining points

Points unassigned after 3 rounds: 87


In [15]:
# ── Visualise all 3 RANSAC lines ──────────────────────────────────────────────
colors    = ['#E63946', '#457B9D', '#2A9D8F']
mk_colors = ['#E63946', '#457B9D', '#2A9D8F']
all_x_cols = [D[:, 0], D[:, 1], D[:, 2]]
all_y_cols = [D[:, 3], D[:, 4], D[:, 5]]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left: raw scatter
ax = axes[0]
for i, (xi, yi) in enumerate(zip(all_x_cols, all_y_cols)):
    ax.scatter(xi, yi, color=colors[i], s=20, alpha=0.6, label=f'Cluster {i+1}')
ax.set_title('Raw Data — 3 Clusters', fontsize=12, fontweight='bold')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.legend(); ax.grid(True, alpha=0.3)

# Right: RANSAC fitted lines
ax = axes[1]
for i, (xi, yi) in enumerate(zip(all_x_cols, all_y_cols)):
    ax.scatter(xi, yi, color=colors[i], s=20, alpha=0.5)

xmin, xmax = X_all.min()-1, X_all.max()+1
for i, (a, b, c) in enumerate(lines):
    xr   = np.linspace(xmin, xmax, 300)
    yr   = (-a*xr + c) / b
    m_l  = -a/b
    q_l  =  c/b
    ax.plot(xr, yr, color=colors[i], lw=2.5,
            label=f'Line {i+1}: y={m_l:.3f}x+({q_l:.3f})')

ax.set_title('RANSAC — 3 Fitted Lines', fontsize=12, fontweight='bold')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
ax.set_ylim(Y_all.min()-2, Y_all.max()+2)

plt.suptitle('Q1(b): RANSAC Line Fitting', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('q1b_ransac.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved q1b_ransac.png")


Saved q1b_ransac.png
